In [0]:
%run ./connectionNotebook

In [0]:
catalog_name = 'adbrag'
target_schema_name = 'gold'
src_schema_name = 'silver'

In [0]:


spark.sql(f"""          

CREATE TABLE IF NOT EXISTS {catalog_name}.{target_schema_name}.customers_gold_scd1
(
    src_CustomerID INT,
    src_CustomerName STRING,
    src_City STRING,
    src_Country STRING,
    src_hash STRING,
    processed_ts TIMESTAMP,
    gold_updated_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_CustomerID)
""")

In [0]:
from pyspark.sql.functions import col, current_timestamp, crc32, concat_ws, coalesce, lit,current_timestamp

silver_df = spark.table(f"{catalog_name}.{src_schema_name}.customers_silver")
#silver_df.show()

In [0]:
src_df = (
    silver_df
    .select(
        col("src_CustomerID"),
        col("src_CustomerName"),
        col("src_City"),
        col("src_Country"),
        col("processed_ts")
    )
    .withColumn(
        "src_hash",
        crc32( #md5
            concat_ws(
                "|",
                coalesce(col("src_CustomerName"), lit("")),
                coalesce(col("src_City"), lit("")),
                coalesce(col("src_Country"), lit(""))
            )
        )
    )
)
   

In [0]:
src_df.createOrReplaceTempView("vw_customers_scd1_source")


In [0]:
spark.sql(f"""
MERGE INTO {catalog_name}.{target_schema_name}.customers_gold_scd1 AS tgt
USING vw_customers_scd1_source AS src
ON tgt.src_CustomerID = src.src_CustomerID

WHEN MATCHED AND tgt.src_hash <> src.src_hash THEN
  UPDATE SET
    tgt.src_CustomerName = src.src_CustomerName,
    tgt.src_City = src.src_City,
    tgt.src_Country = src.src_Country,
    tgt.src_hash = src.src_hash,
    tgt.processed_ts = src.processed_ts,
    tgt.gold_updated_ts = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT
  (
    src_CustomerID,
    src_CustomerName,
    src_City,
    src_Country,
    src_hash,
    processed_ts,
    gold_updated_ts
  )
  VALUES
  (
    src.src_CustomerID,
    src.src_CustomerName,
    src.src_City,
    src.src_Country,
    src.src_hash,
    src.processed_ts,
    current_timestamp()
  )
""")

